# Power Analysis: Waiter's Tips — Solution

**A Waiter's Tips**

The following description was retrieved from the Kaggle page.

> Food servers' tips in restaurants may be influenced by many factors, including the nature of the restaurant, size of the party, and table locations in the restaurant. Restaurant managers need to know which factors matter when they assign tables to food servers. For the sake of staff morale, they usually want to avoid either the substance or the appearance of unfair treatment of the servers, for whom tips (at least in restaurants in the United States) are a major component of pay. In one restaurant, a food server recorded the following data on all customers they served during an interval of two and a half months in early 1990.

Acknowledgements: Bryant, P. G. and Smith, M (1995) *Practical Data Analysis: Case Studies in Business Statistics*. Homewood, IL: Richard D. Irwin Publishing.

The dataset is also available through [Seaborn](https://seaborn.pydata.org/generated/seaborn.load_dataset.html).

**Your task**: for each hypothesis below, run a **power analysis** following the three-step workflow from the [Power Analysis lesson](../../lessons/13_power_analysis.ipynb):

1. **Effect size** — measure how loud the signal is in the data (Cohen's $d$).
2. **Power** — given the sample you actually have, what was your chance of detecting it?
3. **Sample size** — how many observations would you need for 80% power?

After each hypothesis, write one sentence answering: *was this study overpowered, underpowered, or about right?*

Hypotheses to analyze:

- $H_1$: male customers tip more than female customers
- $H_2$: dinner parties tip more than lunch parties
- $H_3$: smokers and non-smokers tip differently
- $H_4$: (... come up with a hypothesis of your own ...)
- **Wrap-up**: which factor would you prioritize for a follow-up study, and how many tables would you need to sample?

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np  # https://numpy.org/doc/
import pandas as pd  # https://pandas.pydata.org/docs/
import seaborn as sns  # https://seaborn.pydata.org/

# statsmodels power analysis — https://www.statsmodels.org/stable/stats.html#power-and-sample-size-calculations
from statsmodels.stats.power import TTestIndPower

# import helpers from the lesson folder
LESSONS_DIR = Path("../../lessons").resolve()
sys.path.insert(0, str(LESSONS_DIR))
from utils import calculate_effect_size, qualify_effect_size

tips = sns.load_dataset("tips")
tips.head()

Column guide:

- `total_bill`: bill amount (food + drinks)
- `tip`: tip amount
- `sex`: customer gender (`Male` / `Female`)
- `smoker`: smoking status (`Yes` / `No`)
- `day`: day of week
- `time`: `Lunch` or `Dinner`
- `size`: party size

## Reminder: the four-way relationship

Effect size, sample size ($n$), significance level ($\alpha$), and power are linked. Fix any three and the fourth is determined.

In `statsmodels`, pass `None` for the unknown in `solve_power(...)`:

| Question | Solve for |
| --- | --- |
| "How many subjects do I need?" | $n$ |
| "What was my chance of finding it?" | power |
| "How small a difference could I have caught?" | effect size |

For **independent two-sample** $t$-tests, use `TTestIndPower` — [docs](https://www.statsmodels.org/stable/generated/statsmodels.stats.power.TTestIndPower.html).

---

## $H_1$: Male customers tip more than female customers

- **Test type**: independent two-sample $t$-test (`tip` by `sex`)
- **Alternative**: `"larger"` (we expect males to tip *more*)

In [ ]:
male_tips = tips.loc[tips["sex"] == "Male", "tip"]
female_tips = tips.loc[tips["sex"] == "Female", "tip"]

### Step 1. Calculate effect size

In [ ]:
n_male = len(male_tips)
n_female = len(female_tips)

effect_size = calculate_effect_size(male_tips, female_tips)

print(f"n observed = {n_male} males and {n_female} females")
print(f"mean tip: ${male_tips.mean():.2f} males vs ${female_tips.mean():.2f} females")
print(f"effect size = {effect_size:.2f}")
print(f"qualified effect size = {qualify_effect_size(abs(effect_size), n_groups=2)}")

### Step 2. Solve for power

In [ ]:
analysis = TTestIndPower()

power = analysis.solve_power(
    effect_size=abs(effect_size),  # magnitude only; direction is in alternative
    nobs1=n_male,                  # group-1 sample size
    ratio=n_female / n_male,       # group-2 size relative to group-1
    alpha=0.05,
    power=None,                    # <-- solving for power
    alternative="larger",
)

print(f"observed power = {power * 100:.1f}%")

### Step 3. Minimum sample size for 80% power

In [ ]:
n_per_group = analysis.solve_power(
    effect_size=abs(effect_size),
    nobs1=None,           # <-- solving for n
    ratio=1.0,            # assume equal-sized groups when planning
    alpha=0.05,
    power=0.80,
    alternative="larger",
)

print(f"n per group needed for 80% power = {np.ceil(n_per_group):.0f}")
print(f"(we observed {n_male} males / {n_female} females — well short of what's needed)")

### Interpretation

*Was this study overpowered, underpowered, or about right for $H_1$? Why?*

**Underpowered.** The effect is small ($d \approx 0.19$) and observed power is only ~40%. We would need roughly 361 customers per group to reach 80% power — more than double what we collected. A non-significant t-test here would tell us little; we simply did not look hard enough.

---

## $H_2$: Dinner parties tip more than lunch parties

- **Test type**: independent two-sample $t$-test (`tip` by `time`)
- **Alternative**: `"larger"`

In [ ]:
dinner_tips = tips.loc[tips["time"] == "Dinner", "tip"]
lunch_tips = tips.loc[tips["time"] == "Lunch", "tip"]

### Step 1. Calculate effect size

In [ ]:
n_dinner = len(dinner_tips)
n_lunch = len(lunch_tips)

effect_size_h2 = calculate_effect_size(dinner_tips, lunch_tips)

print(f"n observed = {n_dinner} dinner and {n_lunch} lunch")
print(f"mean tip: ${dinner_tips.mean():.2f} dinner vs ${lunch_tips.mean():.2f} lunch")
print(f"effect size = {effect_size_h2:.2f}")
print(f"qualified effect size = {qualify_effect_size(abs(effect_size_h2), n_groups=2)}")

### Step 2. Solve for power

In [ ]:
analysis_h2 = TTestIndPower()

power_h2 = analysis_h2.solve_power(
    effect_size=abs(effect_size_h2),
    nobs1=n_dinner,
    ratio=n_lunch / n_dinner,
    alpha=0.05,
    power=None,
    alternative="larger",
)

print(f"observed power = {power_h2 * 100:.1f}%")

### Step 3. Minimum sample size for 80% power

In [ ]:
n_per_group_h2 = analysis_h2.solve_power(
    effect_size=abs(effect_size_h2),
    nobs1=None,
    ratio=1.0,
    alpha=0.05,
    power=0.80,
    alternative="larger",
)

print(f"n per group needed for 80% power = {np.ceil(n_per_group_h2):.0f}")
print(f"(we observed {n_dinner} dinner / {n_lunch} lunch — close, but lunch is under-sampled)")

### Interpretation

*Was this study overpowered, underpowered, or about right for $H_2$? Why?*

**Underpowered, but the most promising of the lot.** The effect is medium ($d \approx 0.27$) and power is ~60%. We need ~168 per group for 80% power — roughly 336 tables total, which fits within a ~240-table follow-up if we balance lunch and dinner shifts more carefully.

---

## $H_3$: Smokers and non-smokers tip differently

- **Test type**: independent two-sample $t$-test (`tip` by `smoker`)
- **Alternative**: `"two-sided"` (we did not pre-commit to a direction)

In [ ]:
smoker_tips = tips.loc[tips["smoker"] == "Yes", "tip"]
nonsmoker_tips = tips.loc[tips["smoker"] == "No", "tip"]

### Step 1. Calculate effect size

In [ ]:
n_smoker = len(smoker_tips)
n_nonsmoker = len(nonsmoker_tips)

effect_size_h3 = calculate_effect_size(smoker_tips, nonsmoker_tips)

print(f"n observed = {n_smoker} smokers and {n_nonsmoker} non-smokers")
print(f"mean tip: ${smoker_tips.mean():.2f} smokers vs ${nonsmoker_tips.mean():.2f} non-smokers")
print(f"effect size = {effect_size_h3:.2f}")
print(f"qualified effect size = {qualify_effect_size(abs(effect_size_h3), n_groups=2)}")

### Step 2. Solve for power

In [ ]:
analysis_h3 = TTestIndPower()

power_h3 = analysis_h3.solve_power(
    effect_size=abs(effect_size_h3),
    nobs1=n_smoker,
    ratio=n_nonsmoker / n_smoker,
    alpha=0.05,
    power=None,
    alternative="two-sided",
)

print(f"observed power = {power_h3 * 100:.1f}%")

### Step 3. Minimum sample size for 80% power

In [ ]:
n_per_group_h3 = analysis_h3.solve_power(
    effect_size=abs(effect_size_h3),
    nobs1=None,
    ratio=1.0,
    alpha=0.05,
    power=0.80,
    alternative="two-sided",
)

print(f"n per group needed for 80% power = {np.ceil(n_per_group_h3):.0f}")
print("(the effect is essentially zero — detecting it would require an impractical sample)")

### Interpretation

*If a t-test reported "no significant difference" for $H_3$, would that be convincing? Use your power result to explain.*

**No — and the study is severely underpowered.** The effect is negligible ($d \approx 0.01$) and observed power is only ~5% (barely above the false-positive rate). A "no significant difference" headline here means *we did not look hard enough to detect even a tiny difference*, not that smoking status has no effect on tipping. This is the Adelie-vs-Chinstrap lesson in reverse: the signal is whisper-quiet.

---

## $H_4$: Your own hypothesis

Pick any comparison in `tips` that interests you. Examples:

- party `size` differs between lunch and dinner
- tip *rate* (`tip / total_bill`) differs by `sex`
- `total_bill` differs between smokers and non-smokers

State your hypothesis, choose the right test type, then run all three power-analysis steps.

**My hypothesis**: party size differs between lunch and dinner visits.

**Test type**: independent two-sample $t$-test (`size` by `time`)

**Alternative**: `"two-sided"` (no pre-committed direction)

In [ ]:
lunch_size = tips.loc[tips["time"] == "Lunch", "size"]
dinner_size = tips.loc[tips["time"] == "Dinner", "size"]

n_lunch_size = len(lunch_size)
n_dinner_size = len(dinner_size)

effect_size_h4 = calculate_effect_size(lunch_size, dinner_size)

print(f"n observed = {n_lunch_size} lunch and {n_dinner_size} dinner")
print(f"mean party size: {lunch_size.mean():.2f} lunch vs {dinner_size.mean():.2f} dinner")
print(f"effect size = {effect_size_h4:.2f}")
print(f"qualified effect size = {qualify_effect_size(abs(effect_size_h4), n_groups=2)}")

analysis_h4 = TTestIndPower()

power_h4 = analysis_h4.solve_power(
    effect_size=abs(effect_size_h4),
    nobs1=n_lunch_size,
    ratio=n_dinner_size / n_lunch_size,
    alpha=0.05,
    power=None,
    alternative="two-sided",
)
print(f"observed power = {power_h4 * 100:.1f}%")

n_per_group_h4 = analysis_h4.solve_power(
    effect_size=abs(effect_size_h4),
    nobs1=None,
    ratio=1.0,
    alpha=0.05,
    power=0.80,
    alternative="two-sided",
)
print(f"n per group needed for 80% power = {np.ceil(n_per_group_h4):.0f}")

### Interpretation

*What did power analysis reveal that a p-value alone would not?*

A t-test might flag a significant difference in party size, but power analysis shows we only had ~36% chance of catching a medium effect ($d \approx 0.23$). We would need ~296 per group for 80% power. The p-value answers "is there a difference?"; power answers "could we have reliably found one?" — and here the answer is no.

---

## Wrap-up: planning the next study

The restaurant manager can only observe **one server** for another two and a half months (~240 tables). Based on your results:

1. Which hypothesis is worth pursuing with this sample budget?
2. Which hypothesis would need a much larger study (or should be dropped)?
3. What sample size would you recommend if the manager wants 80% power on their top priority?

1. **$H_2$ (dinner vs lunch tips)** is the best fit for ~240 tables. It has a medium effect and needs ~168 per group (~336 total) for 80% power — achievable if the manager balances lunch and dinner shifts instead of letting dinner dominate the sample.
2. **$H_3$ (smoker tipping)** should be dropped. The effect is essentially zero; reaching 80% power would require over 100,000 customers per group. **$H_1$ (sex)** also exceeds the budget (~361 per group).
3. For the top priority ($H_2$), recommend **~170 tables per meal period** (lunch and dinner), for **~340 tables total** at 80% power.